# 第一阶段 · Notebook 1：量子计算基础与概念映射

> **配合阅读**：RevModPhys.92.015003 第I–III章  
> **前置知识**：DFT/量子化学基础，熟悉哈密顿量、基组、Slater行列式

---

## 本节目标
1. 建立量子计算与经典量子化学之间的**概念双语字典**
2. 理解量子比特、量子门、量子电路的数学基础
3. 理解**第二量子化**表示和**Jordan-Wigner变换**
4. 为后续Qiskit实践搭建概念框架

**运行环境**：请使用 conda 环境 **`qc_chem`**（含 `numpy` / `scipy`）。在 Cursor 中打开本笔记本后，点击右上角 **选择内核** → 选 **Python (qc_chem)**；若列表中没有，先执行 `Python: Select Interpreter`（命令面板 `Ctrl+Shift+P`）并选中 `...\anaconda3\envs\qc_chem\python.exe`，再重载窗口。

---
## 1. 概念双语字典：从量子化学到量子计算

| 量子化学概念 | 量子计算对应概念 | 说明 |
|------------|----------------|------|
| 自旋轨道（spin-orbital） | 量子比特（qubit） | 一个自旋轨道占据/空置 → 一个量子比特 |0⟩/|1⟩ |
| Slater行列式 | 计算基态（computational basis state） | |0110...⟩ 表示特定电子构型 |
| FCI波函数（CI展开） | 量子态叠加（superposition） | 多个行列式的线性组合 |
| 哈密顿量期望值 ⟨ψ\|H\|ψ⟩ | 泡利算符期望值测量 | 电路末端测量+经典后处理 |
| CCSD(T)参数优化 | 变分参数θ优化（VQE） | 经典优化器调整量子电路参数 |
| 活性空间（active space） | 逻辑量子比特数 | CAS(n,m) ↔ m个量子比特 |
| 基组（basis set） | 编码方式（JW/BK/parity） | 影响量子比特数和门深度 |

**关键洞察**：你在DFT/FCI中熟悉的哈密顿量 $\hat{H}$ 需要被「翻译」成量子比特操作（泡利算符串），这个翻译过程就是**量子化学算法的核心工程步骤**。

---
## 2. 量子比特的数学基础

### 2.1 量子态（State Vector）

单量子比特的一般态：
$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \quad |\alpha|^2 + |\beta|^2 = 1$$

矩阵表示：
$$|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}, \quad |1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$$

**化学类比**：$|0\rangle$ = 轨道空置，$|1\rangle$ = 轨道被占据

### 2.2 多量子比特态

两量子比特的张量积（tensor product）：
$$|\psi\rangle = |\psi_1\rangle \otimes |\psi_2\rangle$$

例：$|01\rangle = |0\rangle \otimes |1\rangle = \begin{pmatrix}0\\1\\0\\0\end{pmatrix}$

**化学类比**：4个自旋轨道的电子构型 $|0110\rangle$ = 轨道1空置、轨道2占据、轨道3占据、轨道4空置

In [1]:
import numpy as np

# 单量子比特基态
ket_0 = np.array([1, 0])
ket_1 = np.array([0, 1])

# 张量积：构造两电子四轨道的行列式基态
# |0110⟩ = |0⟩⊗|1⟩⊗|1⟩⊗|0⟩
state_0110 = np.kron(np.kron(np.kron(ket_0, ket_1), ket_1), ket_0)
print(f"|0110⟩ 维度: {state_0110.shape}")
print(f"|0110⟩ 向量: {state_0110}")
print(f"这代表4个自旋轨道中，轨道1和4空置，轨道2和3被占据")

# FCI波函数：多个行列式的叠加（类比量子态叠加）
# |ψ_FCI⟩ = c1|1100⟩ + c2|0110⟩ + c3|1001⟩ + ...
state_1100 = np.kron(np.kron(np.kron(ket_1, ket_1), ket_0), ket_0)
state_0011 = np.kron(np.kron(np.kron(ket_0, ket_0), ket_1), ket_1)

# Hartree-Fock参考态（主导构型）
c_HF = 0.9939
c_double_excitation = -0.1116
psi_FCI = c_HF * state_1100 + c_double_excitation * state_0011
psi_FCI_normalized = psi_FCI / np.linalg.norm(psi_FCI)
print(f"\nH₂ FCI近似波函数（STO-3G，平衡键长）:")
print(f"  主导HF构型系数: {c_HF:.4f}")
print(f"  双激发构型系数: {c_double_excitation:.4f}")
print(f"  这正是VQE要优化的量子态！")

|0110⟩ 维度: (16,)
|0110⟩ 向量: [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
这代表4个自旋轨道中，轨道1和4空置，轨道2和3被占据

H₂ FCI近似波函数（STO-3G，平衡键长）:
  主导HF构型系数: 0.9939
  双激发构型系数: -0.1116
  这正是VQE要优化的量子态！


---
## 3. 量子门（Quantum Gates）

### 阅读路线（建议顺序）

| 步骤 | 小节 | 你要抓住什么 |
|:---:|:---|:---|
| 1 | **3.1** | 门 = **酉变换**，作用在态矢量上 |
| 2 | **3.2** | **泡利** $I,X,Y,Z$ = 写哈密顿量的「字母」→ 与量子化学直接对接 |
| 3 | **3.3** | 常用 **单比特门**（含是否 Clifford）一张表扫完 |
| 4 | **3.4** | **CNOT** = 双比特门、纠缠、线路里怎么画 |
| 5 | **3.5** | **Clifford** 是什么、为何需要 **+T** |
| 6 | **3.6** | **NISQ（VQE）** 与 **容错编译** 两套说法怎么对应 |

**结构示意图（可选读）**：下面是一段 **Mermaid** 流程图——用**纯文本**写节点与箭头，在支持 Mermaid 的环境（部分 Jupyter、VS Code 的 Markdown 预览插件、GitHub 等）里会画成图。**若你看到的是带 `flowchart` 字样的代码块而不是图，可跳过**，与上方「阅读路线」表说的是同一件事。

```mermaid
flowchart LR
  subgraph chemLink [与量子化学]
    PauliLetters[泡利串]
    Ham[分子哈密顿量]
    PauliLetters --> Ham
  end
  subgraph gates [量子门]
    OneQ[单比特酉门]
    CNOT[CNOT双比特]
    OneQ --> State[改变态矢量]
    CNOT --> Entangle[纠缠比特]
  end
  subgraph resource [资源与通用性]
    Clif[Clifford子集]
    Tgate[T门等非Clifford]
    Clif --> Univ[Clifford加T近似通用]
    Tgate --> Univ
  end
  Ham --> OneQ
  OneQ --> Clif
```

**已安装插件时**：在 Cursor / VS Code 中请执行 **命令面板 → `Developer: Reload Window`** 重载窗口；打开本 Notebook 后，用 **预览**（或 Markdown 单元格的渲染视图）查看上方代码块，应显示为流程图。

**不依赖 Mermaid 的同一结构（SVG，多数环境可直接显示）：**

<svg xmlns='http://www.w3.org/2000/svg' width='760' height='180' font-family='Segoe UI,Microsoft YaHei,sans-serif' font-size='12'><defs><marker id='mrA' markerWidth='8' markerHeight='8' refX='7' refY='4' orient='auto'><path d='M0,0 L8,4 L0,8 Z' fill='#444'/></marker></defs><rect x='15' y='40' width='220' height='95' fill='#e3f2fd' stroke='#1565c0' stroke-width='1.5' rx='6'/><text x='125' y='68' text-anchor='middle' fill='#0d47a1' font-weight='bold' font-size='13'>与量子化学</text><text x='125' y='92' text-anchor='middle' fill='#333'>泡利串 → 分子哈密顿量</text><line x1='235' y1='87' x2='275' y2='87' stroke='#444' stroke-width='2' marker-end='url(#mrA)'/><rect x='285' y='40' width='220' height='95' fill='#e8f5e9' stroke='#2e7d32' stroke-width='1.5' rx='6'/><text x='395' y='68' text-anchor='middle' fill='#1b5e20' font-weight='bold' font-size='13'>量子门</text><text x='395' y='92' text-anchor='middle' fill='#333'>单比特门 / CNOT 纠缠</text><line x1='505' y1='87' x2='545' y2='87' stroke='#444' stroke-width='2' marker-end='url(#mrA)'/><rect x='555' y='40' width='190' height='95' fill='#fff3e0' stroke='#ef6c00' stroke-width='1.5' rx='6'/><text x='650' y='68' text-anchor='middle' fill='#e65100' font-weight='bold' font-size='13'>资源与通用性</text><text x='650' y='92' text-anchor='middle' fill='#333'>Clifford + T</text><text x='650' y='110' text-anchor='middle' fill='#555' font-size='11'>近似通用</text></svg>

---

### 3.1 量子门在做什么？

- **量子门**用 **酉矩阵** $U$（$U^\dagger U = I$）描述，作用在态矢量上：$\lvert\psi\rangle \mapsto U\lvert\psi\rangle$（可连续施加多个门 = 矩阵相乘）。
- **单比特门**：$2\times 2$ 酉矩阵，只动一条量子线。
- **双比特门**（如 **CNOT**）：$4\times 4$ 酉矩阵，关联两条线——化学里多电子 **纠缠 / 关联** 常通过这类门进入线路表示。

---

### 3.2 泡利矩阵：量子化学的「字母表」

第二量子化 + JW 等映射之后，分子哈密顿量写成 **泡利算符的串**（张量积）。因此 $I,X,Y,Z$ 是整套语言的**原子**：

$$X = \sigma_x = \begin{pmatrix}0&1\\1&0\end{pmatrix}, \quad Y = \sigma_y = \begin{pmatrix}0&-i\\i&0\end{pmatrix}, \quad Z = \sigma_z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}$$

**关键性质**：任一 $2\times 2$ **厄米**矩阵都可唯一写成 $I,X,Y,Z$ 的实系数线性组合；多比特上则是 $\{I,X,Y,Z\}^{\otimes N}$ 中元素的线性组合：

$$\hat{H} = \sum_k h_k P_k, \quad P_k \in \{I, X, Y, Z\}^{\otimes N}$$

**与「门」的关系**：$X,Y,Z$ **本身**也是（相差相位的）酉门；下面 **3.3** 里的 **X 门**与这里的 $\sigma_x$ 是同一个矩阵。后面 **Clifford** 的定义，就是「在共轭下把泡利串仍变成泡利串」——所以**先理解泡利，再读 Clifford**最顺。

---

### 3.3 常用单比特门速查

| 门 | 矩阵 | Clifford？ | 典型用途（直觉） |
|:---|:---|:---:|:---|
| **I** | $\begin{pmatrix}1&0\\0&1\end{pmatrix}$ | 是 | 恒等；占位 |
| **X**（NOT） | $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ | 是 | 翻转 $\lvert0\rangle\leftrightarrow\lvert1\rangle$；与泡利 $\sigma_x$ 相同 |
| **H**（Hadamard） | $\frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ | 是 | 产生叠加；常在初态制备里出现 |
| **S**（$\sqrt{Z}$） | $\begin{pmatrix}1&0\\0&i\end{pmatrix}$ | 是 | $S^2=Z$；对 $\lvert1\rangle$ 乘 $i$ |
| **$R_z(\theta)$** | $\begin{pmatrix}e^{-i\theta/2}&0\\0&e^{i\theta/2}\end{pmatrix}$ | 仅当 $\theta$ 为 $\pi/2$ 的整数倍等离散角时 | **NISQ / VQE** 里最常见的可调参数门之一 |
| **T**（$\pi/8$） | $\begin{pmatrix}1&0\\0&e^{i\pi/4}\end{pmatrix}$ | **否** | $R_z(\pi/2)$ 的固定角；**容错**里常与 Clifford 搭配，用 **T 深度 / T 计数** 衡量代价 |

> **两套语言**：**VQE** 论文里常写连续 $R_y(\theta), R_z(\theta)$；**容错 / 编译** 论文里常把任意单比特旋转分解为 **Clifford + T**。同一条线路，只是「计数单位」不同。

---

### 3.4 双比特门：CNOT（受控非）

**作用**：控制线（control）为 $\lvert1\rangle$ 时，对目标线（target）施加 **X**；控制为 $\lvert0\rangle$ 时目标不变。可生成 **Bell 态**（最大纠缠），是多比特线路的**标准纠缠源**。

$$\mathrm{CNOT} = \begin{pmatrix}1&0&0&0\\0&1&0&0\\0&0&0&1\\0&0&1&0\end{pmatrix}\quad\text{（基矢顺序：}\lvert00\rangle,\lvert01\rangle,\lvert10\rangle,\lvert11\rangle\text{；左位为控制）}$$

**与化学**：多自旋轨道的 JW 映射后，哈密顿量中的**相互作用项**在编译成线路时，往往通过 **CNOT + 单比特旋转** 实现泡利项的测量与演化。

---

### 3.5 Clifford 门组（与上表对齐）

**一句话定义**：$n$ 比特 **Pauli 群** $\mathcal{P}_n$ 上，幺正门 $U$ 若满足「对任意 $P\in\mathcal{P}_n$，$U P U^\dagger$ 仍是 Pauli 串（允许整体相位）」，则 $U$ 属于 **Clifford 群**。

**你要记住的三点**：

1. **常用生成元**：单比特 **H、S**（以及 Pauli **X,Y,Z**），加上双比特 **CNOT**，可生成整个 Clifford 群（具体集合依文献略有等价表述）。
2. **Gottesman–Knill**：**只有** Clifford + 计算基测量的线路，经典可高效模拟 → 单靠 Clifford **不够**实现任意指数加速。
3. **Clifford + T**：加上 **T**（或任一非 Clifford 单比特门）后，可 **近似通用** 量子计算；这是容错线路里 **T 计数** 的来源。

---

### 3.6 NISQ 量子化学 vs 容错编译（对照）

| 侧面 | NISQ / VQE（你现在最常碰到的） | 容错 QC / 编译论文 |
|:---|:---|:---|
| 单比特旋转 | 连续 $R_z(\theta)$、$R_y(\theta)$ 作变分参数 | 离散分解为 **Clifford + T** |
| 纠缠 | **CNOT**（或硬件原生等价门） | 同上，但强调 **magic state / T** 资源 |
| 哈密顿量 | 已是 **泡利串**；测 $\langle P_k\rangle$ | 同一套泡利分解；额外关心 **总深度、T 数** |

下一格代码单元用泡利分解演示：任意 $2\times 2$ 厄米矩阵如何用 $I,X,Y,Z$ 展开——与 **3.2** 直接呼应。


In [2]:
import numpy as np

# 泡利矩阵定义
I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])

# 演示：任意2x2厄米矩阵可分解为泡利矩阵
# H_1qubit = a*I + b*X + c*Y + d*Z
# 系数公式：a = Tr(H)/2, b = Tr(XH)/2, c = Tr(YH)/2, d = Tr(ZH)/2

def decompose_to_pauli(H):
    """将单量子比特厄米矩阵分解为泡利矩阵"""
    paulis = {'I': I, 'X': X, 'Y': Y, 'Z': Z}
    coeffs = {}
    for name, P in paulis.items():
        coeff = np.trace(P @ H) / 2
        # 下面两行各司其职：阈值=略去数值上≈0的项；np.real=去掉浮点带来的微小虚部
        if abs(coeff) > 1e-12:
            coeffs[name] = np.real(coeff)  # H、P 厄米 ⇒ Tr(PH)/2 严格为实；取实部只为数值卫生
    return coeffs

# 示例：任取一个 2×2 厄米矩阵，演示 decompose_to_pauli（非某文献「唯一」参数）
# 可粗略类比「两能级 + 耦合」：对角像在位能，非对角像跃迁/杂化；标准单格点 Hubbard（两自旋）通常是 4×4
H_simple = np.array([[-1.0, -0.5],
                     [-0.5,  1.0]])

coeffs = decompose_to_pauli(H_simple)
print("H = " + " + ".join([f"{v:.3f}·{k}" for k, v in coeffs.items()]))

# 验证重构
H_reconstructed = sum(v * {'I':I,'X':X,'Y':Y,'Z':Z}[k] for k, v in coeffs.items())
print(f"\n重构误差: {np.max(np.abs(H_simple - H_reconstructed)):.2e}")
print("\n这就是量子化学哈密顿量'泡利分解'的核心原理！")
print("VQE计算期望值 ⟨ψ|H|ψ⟩ = Σ h_k·⟨ψ|P_k|ψ⟩")

H = -0.500·X + -1.000·Z

重构误差: 0.00e+00

这就是量子化学哈密顿量'泡利分解'的核心原理！
VQE计算期望值 ⟨ψ|H|ψ⟩ = Σ h_k·⟨ψ|P_k|ψ⟩


---
## 4. 第二量子化与Jordan-Wigner变换

### 4.1 第二量子化回顾

你在量子化学中熟悉的哈密顿量（第二量子化形式）：

$$\hat{H} = \sum_{pq} h_{pq} \hat{a}_p^\dagger \hat{a}_q + \frac{1}{2}\sum_{pqrs} g_{pqrs} \hat{a}_p^\dagger \hat{a}_q^\dagger \hat{a}_r \hat{a}_s$$

其中：
- $h_{pq}$：单电子积分（动能 + 核-电子吸引）
- $g_{pqrs}$：双电子排斥积分
- $\hat{a}_p^\dagger, \hat{a}_p$：费米子产生/湮灭算符

**问题**：量子计算机只能操作量子比特（玻色子算符），不能直接处理费米子算符。

### 4.2 Jordan-Wigner（JW）变换

JW变换将费米子算符映射到量子比特泡利算符：

$$\hat{a}_j^\dagger = \frac{1}{2}(X_j - iY_j) \otimes Z_{j-1} \otimes \cdots \otimes Z_0$$

$$\hat{a}_j = \frac{1}{2}(X_j + iY_j) \otimes Z_{j-1} \otimes \cdots \otimes Z_0$$

**直觉理解**：
- $X_j - iY_j$ 部分：改变轨道 j 的占据状态（产生/湮灭电子）
- $Z_0 \otimes \cdots \otimes Z_{j-1}$ 部分：**弦算符（string operator）**，确保费米子的反对易关系（Pauli exclusion principle）

In [4]:
import numpy as np
from itertools import product

# 手动实现Jordan-Wigner变换（教学目的，实际用Qiskit Nature）

# 泡利矩阵
I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])

def jordan_wigner_creation(j, n_qubits):
    """
    构造第j个轨道的JW产生算符（n_qubits个自旋轨道）
    a†_j = Z_0 ⊗ Z_1 ⊗ ... ⊗ Z_{j-1} ⊗ (X_j - iY_j)/2 ⊗ I_{j+1} ⊗ ... ⊗ I_{N-1}
    """
    ops = []
    for k in range(n_qubits):
        if k < j:
            ops.append(Z)           # 弦算符：确保反对易关系
        elif k == j:
            ops.append((X - 1j * Y) / 2)  # 产生算符部分
        else:
            ops.append(I)           # 恒等
    
    # 张量积
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result

def jordan_wigner_annihilation(j, n_qubits):
    return jordan_wigner_creation(j, n_qubits).conj().T

# 验证费米子反对易关系 {a_i, a_j†} = δ_{ij}
n = 4  # 4个自旋轨道（对应H₂的STO-3G基组：2个空间轨道×2自旋）

print("验证JW变换保持费米子反对易关系 {a_i, a_j†} = δ_{ij}")
print("="*60)
for i in range(n):
    for j in range(n):
        a_i = jordan_wigner_annihilation(i, n)
        a_dag_j = jordan_wigner_creation(j, n)
        anticommutator = a_i @ a_dag_j + a_dag_j @ a_i
        expected = np.eye(2**n) if i == j else np.zeros((2**n, 2**n))
        error = np.max(np.abs(anticommutator - expected))
        if error > 1e-10:
            print(f"  {i},{j}: 误差 = {error:.2e} ← 违反！")
        
print("✓ 所有 {a_i, a_j†} = δ_{ij} 关系满足（误差<1e-10）")
print()
print("这意味着JW变换正确地将费米子反对易关系编码到量子比特中")
print("Pauli exclusion principle 自动满足！")

验证JW变换保持费米子反对易关系 {a_i, a_j†} = δ_{ij}
✓ 所有 {a_i, a_j†} = δ_{ij} 关系满足（误差<1e-10）

这意味着JW变换正确地将费米子反对易关系编码到量子比特中
Pauli exclusion principle 自动满足！


---
## 5. H₂分子哈密顿量的泡利分解（完整示例）

H₂分子在STO-3G基组下有4个自旋轨道，经JW变换后得到量子比特哈密顿量：

$$H_{H_2} = g_0 I + g_1 Z_0 + g_2 Z_1 + g_3 Z_0 Z_1 + g_4 Y_0 Y_1 + g_5 X_0 X_1$$

（键长约0.735 Å时，经过粒子守恒约化到2量子比特后的形式）

其中系数 $g_i$ 由单电子积分和双电子积分计算得到。

In [5]:
import numpy as np

# H₂分子在平衡键长（0.735 Å）的泡利哈密顿量系数（STO-3G，JW变换后约化到2量子比特）
# 数据来源：Peruzzo et al. (2014) Nature Communications，VQE首篇论文

# 泡利矩阵
I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])

# H₂ 泡利系数（单位：Hartree）
g = {
    'II': -1.8505,  # 常数项（核排斥 + 单/双电子积分贡献）
    'IZ': +0.3980,
    'ZI': -0.3980,
    'ZZ': +0.0112,
    'XX': +0.1807,
    'YY': +0.1807,
}

def pauli_str_to_matrix(pauli_str):
    """将泡利字符串转为矩阵（如'XZ' → X⊗Z）"""
    map_ = {'I': I, 'X': X, 'Y': Y, 'Z': Z}
    result = map_[pauli_str[0]]
    for p in pauli_str[1:]:
        result = np.kron(result, map_[p])
    return result

# 构造完整哈密顿量矩阵
H_H2 = sum(coeff * pauli_str_to_matrix(pstr) for pstr, coeff in g.items())

print("H₂分子量子比特哈密顿量（STO-3G，平衡键长）：")
print("="*50)
for pstr, coeff in g.items():
    print(f"  {coeff:+.4f} · {pstr[0]}₀{pstr[1]}₁")

# 精确对角化（这就是FCI！）
eigenvalues, eigenvectors = np.linalg.eigh(H_H2)
E_FCI = eigenvalues[0]

print(f"\n精确对角化（= FCI）基态能量: {E_FCI:.6f} Hartree")
print(f"文献值（STO-3G FCI）:         -1.8572 Hartree")
print(f"\n基态波函数（计算基展开）:")
gs = eigenvectors[:, 0]
basis_labels = ['|00⟩', '|01⟩', '|10⟩', '|11⟩']
for label, coeff in zip(basis_labels, gs):
    if abs(coeff) > 0.01:
        print(f"  {coeff:+.4f} × {label}")

print("\n注：|01⟩ = HF参考态（轨道0空，轨道1满），|10⟩ = 双激发态")
print("两者的叠加 → 电子关联效应！这正是FCI相对于HF的改进来源")

H₂分子量子比特哈密顿量（STO-3G，平衡键长）：
  -1.8505 · I₀I₁
  +0.3980 · I₀Z₁
  -0.3980 · Z₀I₁
  +0.0112 · Z₀Z₁
  +0.1807 · X₀X₁
  +0.1807 · Y₀Y₁

精确对角化（= FCI）基态能量: -2.735900 Hartree
文献值（STO-3G FCI）:         -1.8572 Hartree

基态波函数（计算基展开）:
  -0.9774+0.0000j × |01⟩
  +0.2115+0.0000j × |10⟩

注：|01⟩ = HF参考态（轨道0空，轨道1满），|10⟩ = 双激发态
两者的叠加 → 电子关联效应！这正是FCI相对于HF的改进来源


---
## 6. 量子电路图与测量

### 6.1 电路符号约定

```
q_0: ─[H]─●─────[Rz(θ)]─ ⟨Z⟩
           │
q_1: ─────[X]──────────── ⟨Z⟩
```

- `[H]`：Hadamard门（创建叠加态）
- `●-[X]`：CNOT门（受控NOT，创建纠缠）
- `[Rz(θ)]`：参数化旋转门（VQE的核心！）
- `⟨Z⟩`：测量Z算符的期望值

### 6.2 变分量子算法的循环

```
初始参数 θ₀
     ↓
量子电路 U(θ) ─→ 制备态 |ψ(θ)⟩
     ↓
测量 ⟨ψ(θ)|H|ψ(θ)⟩ = Σ g_k ⟨P_k⟩
     ↓
经典优化器更新 θ → θ + Δθ
     ↓
收敛？→ 否 → 回到量子电路
         → 是 → 输出 E₀ = ⟨H⟩_min
```

这就是**VQE（Variational Quantum Eigensolver）**的完整流程，下一个notebook将用Qiskit实现它！

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 可视化VQE的核心思想：变分原理
# 对于参数化态 |ψ(θ)⟩ = cos(θ/2)|HF⟩ + sin(θ/2)|double_exc⟩
# E(θ) = ⟨ψ(θ)|H|ψ(θ)⟩ ≥ E_FCI（变分定理）

# H₂哈密顿量（2量子比特，前面已定义的系数）
I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])

g = {'II': -1.8505, 'IZ': +0.3980, 'ZI': -0.3980, 'ZZ': +0.0112, 'XX': +0.1807, 'YY': +0.1807}
pauli_map = {'I': I, 'X': X, 'Y': Y, 'Z': Z}

H_H2 = sum(c * np.kron(pauli_map[s[0]], pauli_map[s[1]]) for s, c in g.items())
E_FCI = np.linalg.eigvalsh(H_H2)[0]

# HF参考态和双激发态（2量子比特）
psi_HF = np.array([0, 1, 0, 0], dtype=complex)        # |01⟩
psi_double = np.array([0, 0, 1, 0], dtype=complex)    # |10⟩

# 单参数ansatz: |ψ(θ)⟩ = cos(θ/2)|01⟩ + sin(θ/2)|10⟩
thetas = np.linspace(0, 2*np.pi, 200)
energies = []
for theta in thetas:
    psi = np.cos(theta/2) * psi_HF + np.sin(theta/2) * psi_double
    E = np.real(psi.conj() @ H_H2 @ psi)
    energies.append(E)

theta_opt = thetas[np.argmin(energies)]
E_VQE = min(energies)

plt.figure(figsize=(9, 5))
plt.plot(thetas, energies, 'b-', linewidth=2, label=r'$E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle$')
plt.axhline(y=E_FCI, color='r', linestyle='--', linewidth=1.5, label=f'FCI/精确对角化: {E_FCI:.4f} Ha')
plt.axhline(y=-1.1175, color='g', linestyle=':', linewidth=1.5, label='HF能量: -1.1175 Ha')
plt.scatter([theta_opt], [E_VQE], color='red', s=100, zorder=5, label=f'VQE最小值: {E_VQE:.4f} Ha')
plt.xlabel(r'参数 $\theta$ (弧度)', fontsize=13)
plt.ylabel('能量 (Hartree)', fontsize=13)
plt.title('VQE变分原理演示：单参数ansatz搜索H₂基态', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('VQE_variational_principle.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nHF能量: -1.1175 Ha")
print(f"VQE最优能量: {E_VQE:.6f} Ha  (θ_opt = {np.degrees(theta_opt):.1f}°)")
print(f"FCI/精确能量: {E_FCI:.6f} Ha")
print(f"相关能回收率: {(E_VQE - (-1.1175))/(E_FCI - (-1.1175))*100:.1f}%")

---
## 7. 知识检验：概念对应练习

请回答以下问题（答案在下方代码块中）：

1. H₂分子STO-3G基组有2个空间轨道，经JW变换需要**多少个**量子比特？
2. 为什么JW变换需要「弦算符」$Z_0 Z_1 \cdots Z_{j-1}$？
3. VQE的变分参数 $\theta$ 对应量子化学中的哪个概念？
4. 为什么电子关联能很难用少量泡利项精确描述？

**答案参考**：

1. **4个量子比特**（2个空间轨道 × 2自旋 = 4个自旋轨道）。但利用粒子守恒和自旋守恒对称性，可以约化到2量子比特。

2. **费米子反对易关系** $\{a_i, a_j^\dagger\} = \delta_{ij}$。量子比特是玻色子型（对易），弦算符引入了必要的相位（$(-1)^{\text{粒子数}}$），确保交换两个费米子时波函数反号（Pauli exclusion principle）。

3. 对应 **CCSD中的振幅参数** $t_1, t_2$（单激发和双激发的权重）。UCCSD ansatz正是将CC振幅作为VQE的变分参数。

4. 电子关联来自大量电子构型的叠加。DFT中用交换关联泛函近似这个效果，而量子计算中需要**测量每个泡利项的期望值**，项数随体系增大而指数增长（但通过分组测量和对称性可大幅减少）。

---
## 8. 本节小结与下一步

本节建立了以下核心概念映射：
- 自旋轨道 → 量子比特
- FCI波函数 → 量子态叠加  
- **§3 量子门**：酉门 → 泡利「字母表」→ 单比特门表（含 Clifford/T）→ CNOT → Clifford+T 与 NISQ/容错两套说法
- 费米子算符 → JW变换 → 泡利算符串
- 哈密顿量期望值 → 泡利项测量求和
- 参数化波函数（CC）→ 参数化量子电路（ansatz）

**下一步**：`02_Qiskit入门实践.ipynb` — 用Qiskit代码实现上述所有概念！

**配合阅读**：
- RevModPhys.92.015003 第II节：Classical quantum chemistry（回顾你的化学背景）
- RevModPhys.92.015003 第III节：Fermion-to-qubit mappings